In [ ]:
 # !pip install transformers

In [ ]:
 # !pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer ,Trainer, TrainingArguments,T5ForConditionalGeneration


In [ ]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

In [ ]:
# random sampling
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

# Data Pre_processing


In [ ]:
import re
def clean_data(text):
    text=re.sub(r"\r\n"," ",text)
    text=re.sub(r"\s+"," ",text)
    text=re.sub(r"<,*?>"," ",text)
    text=text.strip().lower()
    return text

In [ ]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

# tokenize

In [ ]:
!pip install sentencepiece
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [ ]:
# row_data->tokenized inputs for
def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)
    inputs["labels"]=targets["input_ids"]
    return inputs

In [ ]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

#working with our model

In [ ]:
# n
model=T5ForConditionalGeneration.from_pretrained("t5-small")


In [ ]:
# fine tune
import torch
if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

model.to(device)


In [ ]:
# training arguments

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=64,
    save_total_limit=1,
    logging_steps=100,
)

In [ ]:
trainer=Trainer(model=model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=val_dataset,
)



In [ ]:
trainer.train()

C:\Users\sagar\anaconda3\envs\deep\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
from google.colab import files
files.download("saved_summary_model.zip")

FileNotFoundError: Cannot find file: saved_summary_model.zip